In [9]:
import pandas as pd
import numpy as np

df= pd.read_csv("Titanic-Dataset.csv")

print("Initial shape:", df.shape)
print(df.isnull().sum())

df["Age"]= df.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

df["Embarked"]= df["Embarked"].fillna(df["Embarked"].mode()[0])

df["HasCabin"]= df["Cabin"].notnull().astype(int)
df.drop(columns=["Cabin"], inplace=True)

df.drop(columns=["PassengerId", "Ticket", "Name"], inplace=True)

df["FamilySize"]= df["SibSp"] + df["Parch"] + 1
df["IsAlone"]= (df["FamilySize"] == 1).astype(int)

df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

print("\nFinal shape:", df.shape)
print(df.isnull().sum())
print(df.head())

df.to_csv("titanic_cleaned.csv", index=False)
print("\nSaved cleaned dataset as titanic_clean,csv")

Initial shape: (891, 12)
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Final shape: (891, 12)
Survived      0
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
HasCabin      0
FamilySize    0
IsAlone       0
Embarked_Q    0
Embarked_S    0
dtype: int64
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  HasCabin  FamilySize  \
0         0       3    0  22.0      1      0   7.2500         0           2   
1         1       1    1  38.0      1      0  71.2833         1           2   
2         1       3    1  26.0      0      0   7.9250         0           1   
3         1       1    1  35.0      1      0  53.1000         1           2   
4         0       3    0  35.0      0      0   8.0500         0           1   

   IsAlone  Embarked_Q  Embark

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

train_df = df[df['Survived'].notna()].copy()


feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'FamilySize', 'IsAlone', 'Embarked_Q', 'Embarked_S']

X = train_df[feature_cols]
y = train_df['Survived'].astype(int)

# 3. Fill any unexpected missing values with the column median
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# 4. Split into training and validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

# 5. Scale features (Crucial for SVM, optional for Decision Trees)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Data processing complete! Ready to train models.")